# 20: Multi-Source Spatial Data Tabulation

> **Prerequisites**: This notebook presupposes familiarity with Census primitives from
> [NB04](04_Spatial_Data_Census_Boundaries.ipynb) (TIGER, GEOID, crosswalks) and
> [NB15](15_Census_Demographics_Integration.ipynb) (convenience functions). NLRB background
> is in [NB19](19_NLRB_Data_Integration.ipynb).

This notebook demonstrates how to join and tabulate data from **four sources** at shared
geographic levels:

| Source | Module | Join Key | Geographic Levels | Data Type |
|--------|--------|----------|-------------------|-----------|
| Census ACS/TIGER | `geo.census_api_client`, `geo.spatial_data` | GEOID | State, County, Tract, Block Group | Demographics + Boundaries |
| NCES EDGE | `geo.nces_download`, `config.nces_constants` | Geometry (spatial join) | Locale territory, School point | Urbanicity + Education |
| NLRB | `geo.django.services.nlrb_service` | State FIPS | State → Region | Labor regions |
| RDH | `data.redistricting_data_hub` | Geometry (spatial overlay) | District, Precinct | Redistricting plans |

## Table of Contents

1. [Setup](#1-setup)
2. [Shared Geographic Base](#2-shared-geographic-base)
3. [Census × NCES — Urbanicity Analysis](#3-census--nces--urbanicity-analysis)
4. [Census × NLRB — Labor Region Demographics](#4-census--nlrb--labor-region-demographics)
5. [Census × RDH — District Demographic Profiles](#5-census--rdh--district-demographic-profiles)
6. [NCES × RDH — Schools per District by Urbanicity](#6-nces--rdh--schools-per-district-by-urbanicity)
7. [Unified Multi-Source Tabulation](#7-unified-multi-source-tabulation)
8. [Pydantic Schema Validation](#8-pydantic-schema-validation)

For a detailed treatment of margin-of-error propagation and scale considerations,
see `docs/MULTI_SOURCE_TABULATION_GUIDE.md`.

In [ ]:
import os
import warnings
from pathlib import Path

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

import siege_utilities as su

su.configure_shared_logging(level="INFO")

# Census
from siege_utilities.geo.census_api_client import (
    CensusAPIClient,
    get_census_data_with_geometry,
    get_population,
)
from siege_utilities.geo.spatial_data import (
    get_census_boundaries,
    normalize_state_identifier,
)
from siege_utilities.geo.geoid_utils import construct_geoid, parse_geoid
from siege_utilities.config.census_constants import STATE_FIPS_CODES

# NCES
from siege_utilities.geo.nces_download import NCESDownloader
from siege_utilities.config.nces_constants import (
    classify_urbanicity,
    get_locale_category,
    get_locale_subcategory,
    LOCALE_NUMERIC_CODES,
)

# NLRB
from siege_utilities.geo.django.services.nlrb_service import NLRB_REGIONS

# RDH
from siege_utilities.data.redistricting_data_hub import (
    RDHClient,
    fetch_enacted_plan,
    fetch_precinct_results,
    fetch_cvap,
    compute_compactness,
    demographic_profile,
)

# Pydantic schemas
from siege_utilities.geo.schemas.redistricting import (
    RedistrictingPlanSchema,
    PlanDistrictSchema,
    DistrictDemographicsSchema,
)
from siege_utilities.geo.schemas.education import (
    NCESLocaleBoundarySchema,
    SchoolLocationSchema,
)

OUTPUT_DIR = Path("output") / "notebook_20"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
su.log_info(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Credential loading — Census API key + RDH credentials
# Resolution: 1Password → environment variable → None

census_api_key = os.environ.get("CENSUS_API_KEY")
if not census_api_key:
    try:
        import subprocess
        result = subprocess.run(
            ["op", "item", "get", "Census API Key", "--fields", "credential", "--reveal"],
            capture_output=True, text=True, timeout=10,
        )
        if result.returncode == 0 and result.stdout.strip():
            census_api_key = result.stdout.strip()
            os.environ["CENSUS_API_KEY"] = census_api_key
            su.log_info("Census API key loaded from 1Password")
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass

if census_api_key:
    su.log_info("Census API key available")
else:
    su.log_warning("No Census API key — API calls will be rate-limited")

# RDH credentials
RDH_AVAILABLE = False
rdh_user = os.environ.get("RDH_USERNAME")
rdh_pass = os.environ.get("RDH_PASSWORD")
if not (rdh_user and rdh_pass):
    try:
        import subprocess
        for field, env_var in [("username", "RDH_USERNAME"), ("password", "RDH_PASSWORD")]:
            result = subprocess.run(
                ["op", "item", "get", "RDH", "--fields", field, "--reveal"],
                capture_output=True, text=True, timeout=10,
            )
            if result.returncode == 0 and result.stdout.strip():
                os.environ[env_var] = result.stdout.strip()
        rdh_user = os.environ.get("RDH_USERNAME")
        rdh_pass = os.environ.get("RDH_PASSWORD")
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass

if rdh_user and rdh_pass:
    RDH_AVAILABLE = True
    su.log_info("RDH credentials available")
else:
    su.log_warning("RDH credentials unavailable — RDH sections will use Census CD boundaries as proxy")

## Data Source Registry

| Source | Module | Auth | Join Key | Geographic Levels | Data Type |
|--------|--------|------|----------|-------------------|-----------|
| Census ACS | `geo.census_api_client` | API key (optional) | GEOID | State, County, Tract, BG | Demographics |
| Census TIGER | `geo.spatial_data` | None | GEOID | State, County, Tract, BG | Boundaries |
| NCES EDGE | `geo.nces_download` | None | Geometry | Locale territory, School point | Urbanicity |
| NLRB | `geo.django.services.nlrb_service` | None | State FIPS | State → Region | Labor regions |
| RDH | `data.redistricting_data_hub` | Username/password | Geometry | District, Precinct | Plans |

**Join strategies differ by source pair** — see the cross-tabulation summary in Part 8.

## 2. Shared Geographic Base

We use **Virginia** as the running example throughout this notebook. VA is a good choice
because it has:
- A manageable number of counties (133) and tracts (~2,000)
- 11 congressional districts with recent redistricting
- Coverage from all four data sources
- A mix of urban, suburban, and rural areas

In [ ]:
# Download VA county + tract TIGER boundaries
STATE = "VA"
STATE_FIPS = "51"  # Virginia FIPS code

va_counties = get_census_boundaries(
    year=2022,
    geographic_level="county",
    state_identifier=STATE,
)
su.log_info(f"VA counties: {len(va_counties)} features")

va_tracts = get_census_boundaries(
    year=2022,
    geographic_level="tract",
    state_identifier=STATE,
)
su.log_info(f"VA tracts: {len(va_tracts)} features")

va_counties.head()

In [ ]:
# Fetch VA county-level demographics with geometry
# B01003_001E = total population, B19013_001E = median household income
va_county_demo = get_census_data_with_geometry(
    year=2022,
    geography="county",
    variables=["B01003_001E", "B19013_001E"],
    state=STATE,
    include_moe=True,
)
va_county_demo = va_county_demo.rename(columns={
    "B01003_001E": "total_pop",
    "B01003_001M": "total_pop_moe",
    "B19013_001E": "median_income",
    "B19013_001M": "median_income_moe",
})
su.log_info(f"County demographics: {len(va_county_demo)} records, columns: {list(va_county_demo.columns)}")

# Quick choropleth — median household income by county
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
va_county_demo.plot(
    column="median_income",
    ax=ax,
    legend=True,
    cmap="YlOrRd",
    edgecolor="0.5",
    linewidth=0.3,
    legend_kwds={"label": "Median Household Income ($)"},
    missing_kwds={"color": "lightgrey"},
)
ax.set_title("Virginia — Median Household Income by County (ACS 2022)")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "va_county_income.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Census × NCES — Urbanicity Analysis

NCES classifies every location in the US into one of 12 locale types across four
categories: **City**, **Suburban**, **Town**, and **Rural**. We spatial-join VA tracts
to locale boundaries, then crosstab Census demographics by urbanicity.

**Join strategy**: Spatial join — tract centroids intersecting locale boundary polygons.

**MoE note**: When aggregating ACS tract-level estimates by locale category, margins of
error must be propagated. For sums, use the root-sum-of-squares formula:
`MoE_agg = sqrt(sum(MoE_i^2))`. See `docs/MULTI_SOURCE_TABULATION_GUIDE.md` for details.

In [ ]:
# Download NCES locale boundaries (national — 12 locale type polygons)
nces = NCESDownloader()
locale_boundaries = nces.download_locale_boundaries(2023)
su.log_info(f"NCES locale boundaries: {len(locale_boundaries)} records")
locale_boundaries.head()

In [ ]:
# Spatial join — classify VA tracts by NCES locale
# Use tract centroids to get a 1:1 mapping (each tract → one locale)
va_tracts_with_centroid = va_tracts.copy()
va_tracts_with_centroid["centroid"] = va_tracts_with_centroid.geometry.centroid
va_tracts_centroids = va_tracts_with_centroid.set_geometry("centroid")

# Ensure matching CRS
if locale_boundaries.crs != va_tracts_centroids.crs:
    locale_boundaries = locale_boundaries.to_crs(va_tracts_centroids.crs)

tracts_by_locale = gpd.sjoin(
    va_tracts_centroids,
    locale_boundaries[["locale_category", "locale_subcategory", "geometry"]],
    how="left",
    predicate="intersects",
)
# Restore original geometry
tracts_by_locale = tracts_by_locale.set_geometry(va_tracts_with_centroid.geometry)
tracts_by_locale = tracts_by_locale.drop(columns=["centroid"], errors="ignore")

su.log_info(f"Tracts classified by locale: {tracts_by_locale['locale_category'].value_counts().to_dict()}")
tracts_by_locale[["GEOID", "locale_category", "locale_subcategory"]].head(10)

In [ ]:
# Fetch tract-level demographics and merge with locale classification
va_tract_demo = get_census_data_with_geometry(
    year=2022,
    geography="tract",
    variables=["B01003_001E", "B19013_001E"],
    state=STATE,
    include_moe=True,
)
va_tract_demo = va_tract_demo.rename(columns={
    "B01003_001E": "total_pop",
    "B01003_001M": "total_pop_moe",
    "B19013_001E": "median_income",
    "B19013_001M": "median_income_moe",
})

# Merge locale classification onto demographic data
tract_locale_demo = va_tract_demo.merge(
    tracts_by_locale[["GEOID", "locale_category", "locale_subcategory"]],
    on="GEOID",
    how="left",
)
su.log_info(f"Tracts with locale + demographics: {len(tract_locale_demo)}")

In [ ]:
# Cross-tabulation: demographics by locale category
locale_summary = (
    tract_locale_demo
    .dropna(subset=["locale_category"])
    .groupby("locale_category")
    .agg(
        tract_count=("GEOID", "count"),
        total_pop=("total_pop", "sum"),
        # Propagate MoE for sums: sqrt(sum(moe_i^2))
        total_pop_moe=("total_pop_moe", lambda x: np.sqrt((x**2).sum())),
        avg_median_income=("median_income", "mean"),
    )
    .round({"total_pop_moe": 0, "avg_median_income": 0})
    .sort_values("total_pop", ascending=False)
)
display(locale_summary)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

locale_summary["total_pop"].plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Total Population by Locale Category")
axes[0].set_ylabel("Population")
axes[0].tick_params(axis="x", rotation=0)

locale_summary["avg_median_income"].plot.bar(ax=axes[1], color="darkorange")
axes[1].set_title("Average Median Income by Locale Category")
axes[1].set_ylabel("Income ($)")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "census_nces_crosstab.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Download VA school locations and count by locale
va_schools = nces.download_school_locations(2023, state_abbr="VA")
su.log_info(f"VA schools: {len(va_schools)} locations")

schools_by_locale = (
    va_schools
    .groupby("locale_category")
    .size()
    .reset_index(name="school_count")
    .sort_values("school_count", ascending=False)
)
display(schools_by_locale)

## 4. Census × NLRB — Labor Region Demographics

NLRB divides the US into 26 active regions. Each region covers one or more states.
Some states appear in multiple regions (e.g., NY is in regions 2, 3, and 29).

**Join strategy**: State FIPS lookup. Since NLRB regions are state-level aggregations,
we join via state abbreviation → region number using the `NLRB_REGIONS` dict.

**Scale note**: NLRB regions are coarse — they aggregate entire states. Avoid drawing
sub-state conclusions from region-level statistics. The proper interpretation is
"states in this region have a combined population of X."

In [ ]:
# Build state-to-region mapping from NLRB_REGIONS
# Note: some states map to multiple regions
state_to_regions = {}
for region_num, info in NLRB_REGIONS.items():
    for st in info["states"]:
        state_to_regions.setdefault(st, []).append(region_num)

# For aggregation, assign each state to its primary (lowest-numbered) region
state_to_primary_region = {st: min(regions) for st, regions in state_to_regions.items()}

su.log_info(f"States with multiple NLRB regions: "
            f"{[(st, regs) for st, regs in state_to_regions.items() if len(regs) > 1]}")

# Show Virginia's region(s)
va_regions = state_to_regions.get("VA", [])
for r in va_regions:
    su.log_info(f"VA → NLRB Region {r} ({NLRB_REGIONS[r]['office']})")

In [ ]:
# Fetch national state-level population
state_pop = get_population(state="*", geography="state", year=2022)
su.log_info(f"State population data: {len(state_pop)} records")
state_pop.head()

In [ ]:
# Map state FIPS → abbreviation → primary NLRB region
from siege_utilities.config.census_constants import FIPS_TO_STATE

state_pop["state_abbr"] = state_pop["GEOID"].map(
    lambda fips: FIPS_TO_STATE.get(fips, {}).get("abbreviation", "")
)
state_pop["nlrb_region"] = state_pop["state_abbr"].map(state_to_primary_region)

# Aggregate population by NLRB region
# Determine the population column name (varies by Census response)
pop_col = [c for c in state_pop.columns if "pop" in c.lower() or c == "B01003_001E"]
pop_col = pop_col[0] if pop_col else state_pop.columns[-1]

region_pop = (
    state_pop
    .dropna(subset=["nlrb_region"])
    .groupby("nlrb_region")
    .agg(
        total_pop=(pop_col, "sum"),
        state_count=("state_abbr", "count"),
        states=("state_abbr", lambda x: ", ".join(sorted(x))),
    )
    .sort_values("total_pop", ascending=False)
)
region_pop["office"] = region_pop.index.map(lambda r: NLRB_REGIONS[r]["office"])
display(region_pop)

In [ ]:
# Horizontal bar chart of NLRB region populations
fig, ax = plt.subplots(figsize=(10, 8))
region_pop_sorted = region_pop.sort_values("total_pop", ascending=True)
bars = ax.barh(
    [f"Region {int(r)} ({region_pop_sorted.loc[r, 'office']})" for r in region_pop_sorted.index],
    region_pop_sorted["total_pop"],
    color="steelblue",
)
ax.set_xlabel("Total Population")
ax.set_title("Population by NLRB Region (Primary Assignment)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nlrb_region_population.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Census × RDH — District Demographic Profiles

RDH provides enacted redistricting plan boundaries. We overlay Census tract
demographics onto district polygons to build a demographic profile per district.

**Join strategy**: Spatial overlay — tracts are area-weighted into districts using
`demographic_profile()`, which computes population-weighted averages for each district.

**MoE note**: Spatial overlay introduces *areal interpolation* error on top of ACS
sampling error. The combined uncertainty is difficult to quantify precisely —
see `docs/MULTI_SOURCE_TABULATION_GUIDE.md` §4 for guidance.

In [ ]:
# Fetch VA congressional district boundaries
if RDH_AVAILABLE:
    try:
        va_congress = fetch_enacted_plan(state="VA", chamber="congress")
        su.log_info(f"RDH enacted plan: {len(va_congress)} districts")
    except Exception as e:
        su.log_warning(f"RDH fetch failed ({e}), falling back to Census CD boundaries")
        RDH_AVAILABLE = False

if not RDH_AVAILABLE:
    # Fallback: Census congressional district boundaries
    va_congress = get_census_boundaries(
        year=2022,
        geographic_level="congressional district",
        state_identifier=STATE,
    )
    su.log_info(f"Census CD boundaries (proxy): {len(va_congress)} districts")

va_congress.head()

In [ ]:
# Compute compactness scores
# Project to equal-area CRS (EPSG:5070 — NAD83 / Conus Albers) for accurate area calculations
va_congress_ea = va_congress.to_crs(epsg=5070)

compactness = compute_compactness(
    va_congress_ea,
    district_id_col="GEOID",
)
su.log_info(f"Compactness scores for {len(compactness)} districts")
display(compactness)

In [ ]:
# Demographic profile — aggregate tract demographics by district
district_demo = demographic_profile(
    plan_gdf=va_congress,
    census_gdf=va_tract_demo,
    district_id_col="GEOID",
    population_cols=["total_pop", "median_income"],
)
su.log_info(f"District demographic profiles: {len(district_demo)} districts")
display(district_demo)

In [ ]:
# Choropleth — districts colored by median income with compactness annotations
# Merge compactness scores onto the district GeoDataFrame
va_congress_plot = va_congress.merge(compactness, on="GEOID", how="left")
va_congress_plot = va_congress_plot.merge(
    district_demo[["GEOID", "median_income"]],
    on="GEOID",
    how="left",
    suffixes=("", "_profile"),
)

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
va_congress_plot.plot(
    column="median_income",
    ax=ax,
    legend=True,
    cmap="YlGnBu",
    edgecolor="0.3",
    linewidth=0.5,
    legend_kwds={"label": "Median Income ($)"},
    missing_kwds={"color": "lightgrey"},
)

# Annotate with Polsby-Popper compactness score
if "polsby_popper" in va_congress_plot.columns:
    for _, row in va_congress_plot.iterrows():
        centroid = row.geometry.centroid
        pp = row.get("polsby_popper", None)
        if pp is not None:
            ax.annotate(
                f"PP={pp:.2f}",
                xy=(centroid.x, centroid.y),
                ha="center", va="center",
                fontsize=7, fontweight="bold",
                color="black",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
            )

ax.set_title("VA Congressional Districts — Median Income with Compactness (Polsby-Popper)")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "district_income_compactness.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. NCES × RDH — Schools per District by Urbanicity

Spatial join of school point locations onto congressional district polygons,
then crosstab by NCES locale category.

**Join strategy**: Point-in-polygon — `gpd.sjoin(schools, districts, predicate='within')`.

In [ ]:
# Spatial join — schools into congressional districts
if va_schools.crs != va_congress.crs:
    va_schools = va_schools.to_crs(va_congress.crs)

schools_in_districts = gpd.sjoin(
    va_schools,
    va_congress[["GEOID", "geometry"]].rename(columns={"GEOID": "district_geoid"}),
    how="inner",
    predicate="within",
)
su.log_info(f"Schools matched to districts: {len(schools_in_districts)}")

In [ ]:
# Cross-tabulation: schools by district × locale category
district_locale_crosstab = pd.crosstab(
    schools_in_districts["district_geoid"],
    schools_in_districts["locale_category"],
    margins=True,
)
display(district_locale_crosstab)

In [ ]:
# Stacked bar chart — schools per district by urbanicity
plot_data = district_locale_crosstab.drop(index="All", columns="All", errors="ignore")

fig, ax = plt.subplots(figsize=(12, 6))
plot_data.plot.bar(stacked=True, ax=ax, colormap="Set2")
ax.set_xlabel("Congressional District")
ax.set_ylabel("Number of Schools")
ax.set_title("VA Schools by Congressional District and NCES Locale Category")
ax.legend(title="Locale", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "schools_district_locale.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Unified Multi-Source Tabulation

A reusable function that combines data from all four sources at the **county level**.
This is the coarsest common denominator that all sources can be aggregated to.

**Join strategies per source**:
- **Census** → GEOID join (exact match)
- **NCES** → spatial join (school point-in-county polygon), then count per county
- **NLRB** → FIPS lookup (state FIPS → region number)
- **RDH** → spatial join (county centroid in district polygon)

**Scale warning**: Aggregating tract-level data to county level is lossy — within-county
variation is collapsed. Aggregating county centroids to districts is approximate — a
county straddling two districts is assigned to whichever contains its centroid.

In [ ]:
def build_multi_source_tabulation(
    county_gdf: gpd.GeoDataFrame,
    schools_gdf: gpd.GeoDataFrame | None = None,
    locale_boundaries_gdf: gpd.GeoDataFrame | None = None,
    nlrb_regions: dict | None = None,
    district_gdf: gpd.GeoDataFrame | None = None,
    state_fips: str | None = None,
) -> gpd.GeoDataFrame:
    """Combine data from Census, NCES, NLRB, and RDH at the county level.

    Parameters
    ----------
    county_gdf : GeoDataFrame
        County boundaries with demographics (must have GEOID, total_pop, median_income).
    schools_gdf : GeoDataFrame, optional
        School point locations with locale_category column.
    locale_boundaries_gdf : GeoDataFrame, optional
        NCES locale boundary polygons (unused if schools already have locale_category).
    nlrb_regions : dict, optional
        NLRB_REGIONS dict. Defaults to the imported NLRB_REGIONS.
    district_gdf : GeoDataFrame, optional
        Congressional district boundaries for spatial join.
    state_fips : str, optional
        2-digit state FIPS code for NLRB region lookup.

    Returns
    -------
    GeoDataFrame
        County-level table with columns from all available sources.
    """
    result = county_gdf.copy()

    # --- NCES: count schools per county by locale ---
    if schools_gdf is not None:
        schools_proj = schools_gdf.to_crs(result.crs) if schools_gdf.crs != result.crs else schools_gdf
        schools_in_county = gpd.sjoin(
            schools_proj,
            result[["GEOID", "geometry"]],
            how="inner",
            predicate="within",
        )
        school_counts = schools_in_county.groupby("GEOID").size().rename("school_count")
        result = result.merge(school_counts, on="GEOID", how="left")
        result["school_count"] = result["school_count"].fillna(0).astype(int)

        # Count urban vs rural schools
        if "locale_category" in schools_in_county.columns:
            for cat in ["city", "suburban", "town", "rural"]:
                cat_count = (
                    schools_in_county[schools_in_county["locale_category"] == cat]
                    .groupby("GEOID").size()
                    .rename(f"{cat}_schools")
                )
                result = result.merge(cat_count, on="GEOID", how="left")
                result[f"{cat}_schools"] = result[f"{cat}_schools"].fillna(0).astype(int)

    # --- NLRB: state FIPS → region ---
    regions = nlrb_regions or NLRB_REGIONS
    if state_fips:
        state_abbr = FIPS_TO_STATE.get(state_fips, {}).get("abbreviation", "")
        primary_region = state_to_primary_region.get(state_abbr)
        if primary_region:
            result["nlrb_region"] = primary_region
            result["nlrb_office"] = regions[primary_region]["office"]

    # --- RDH: county centroid → district ---
    if district_gdf is not None:
        district_proj = district_gdf.to_crs(result.crs) if district_gdf.crs != result.crs else district_gdf
        county_centroids = result.copy()
        county_centroids["centroid"] = county_centroids.geometry.centroid
        county_centroids = county_centroids.set_geometry("centroid")

        cd_join = gpd.sjoin(
            county_centroids[["GEOID", "centroid"]].set_geometry("centroid"),
            district_proj[["GEOID", "geometry"]].rename(columns={"GEOID": "congressional_district"}),
            how="left",
            predicate="within",
        )
        result = result.merge(
            cd_join[["GEOID", "congressional_district"]],
            on="GEOID",
            how="left",
        )

    return result

In [ ]:
# Execute the unified tabulation on VA data
va_unified = build_multi_source_tabulation(
    county_gdf=va_county_demo,
    schools_gdf=va_schools,
    nlrb_regions=NLRB_REGIONS,
    district_gdf=va_congress,
    state_fips=STATE_FIPS,
)
su.log_info(f"Unified tabulation: {len(va_unified)} counties, {len(va_unified.columns)} columns")

In [ ]:
# Display the unified DataFrame
display_cols = [
    col for col in [
        "NAME", "GEOID", "total_pop", "median_income",
        "school_count", "city_schools", "suburban_schools", "town_schools", "rural_schools",
        "nlrb_region", "nlrb_office", "congressional_district",
    ]
    if col in va_unified.columns
]
va_unified[display_cols].sort_values("total_pop", ascending=False).head(20)

In [ ]:
# Group by congressional district for summary statistics
if "congressional_district" in va_unified.columns:
    district_summary = (
        va_unified
        .dropna(subset=["congressional_district"])
        .groupby("congressional_district")
        .agg(
            county_count=("GEOID", "count"),
            total_pop=("total_pop", "sum"),
            avg_income=("median_income", "mean"),
            total_schools=("school_count", "sum") if "school_count" in va_unified.columns else ("GEOID", "count"),
        )
        .round({"avg_income": 0})
        .sort_values("total_pop", ascending=False)
    )
    display(district_summary)
else:
    su.log_info("No congressional district column — skipping district summary")

## 8. Pydantic Schema Validation

The `siege_utilities` Pydantic schemas can validate cross-source data records
without requiring Django. This is useful for ensuring data quality before
loading into a database or passing to downstream pipelines.

In [ ]:
# Validate a DistrictDemographicsSchema instance
# Demonstrates how Census variable codes map to the schema's named fields
demo_record = DistrictDemographicsSchema(
    district_id="VA-01",
    dataset="acs5",
    year=2022,
    pop_white=450000,
    pop_black=120000,
    pop_hispanic=85000,
    pop_asian=45000,
    pop_native=3000,
    pop_other=8000,
    pop_two_or_more=25000,
    median_household_income=72500,
    poverty_rate=11.3,
    values={
        "B01003_001E": 736000,
        "B19013_001E": 72500,
        "B17001_002E": 83168,
    },
)

su.log_info(f"Schema validated: {demo_record.district_id}")
su.log_info(f"Race fields auto-populated: white={demo_record.pop_white}, "
            f"black={demo_record.pop_black}, hispanic={demo_record.pop_hispanic}")
su.log_info(f"Raw Census values: {demo_record.values}")

# Validate an NCESLocaleBoundarySchema
locale_record = NCESLocaleBoundarySchema(
    geoid="11",
    name="City, Large",
    locale_code=11,
    locale_category="city",
    locale_subcategory="city_large",
    nces_year=2023,
    vintage_year=2023,
)
su.log_info(f"NCES schema validated: {locale_record.name} (code={locale_record.locale_code})")

# Validate a SchoolLocationSchema
school_record = SchoolLocationSchema(
    geoid="510001000001",
    name="Example School",
    ncessch="510001000001",
    school_name="Example School",
    lea_id="5100010",
    locale_code=21,
    locale_category="suburban",
    locale_subcategory="suburb_large",
    vintage_year=2023,
)
su.log_info(f"School schema validated: {school_record.school_name} "
            f"(locale={school_record.locale_category})")

## Summary

This notebook demonstrated cross-source spatial tabulation using four data sources
available in `siege_utilities`:

### Cross-Tabulations Demonstrated

| Cross-Tab | Join Strategy | Join Key | Geographic Level |
|-----------|--------------|----------|------------------|
| Census × NCES | Spatial join (tract centroid in locale boundary) | Geometry | Tract → Locale |
| Census × NLRB | FIPS lookup | State FIPS | State → Region |
| Census × RDH | Spatial overlay (tract in district) | Geometry | Tract → District |
| NCES × RDH | Spatial join (school point in district) | Geometry | School → District |
| All Sources | Unified county-level tabulation | GEOID + spatial | County |

### Key Functions

| Function | Source | Purpose |
|----------|--------|---------|
| `get_census_data_with_geometry()` | Census | Demographics + boundaries in one call |
| `get_census_boundaries()` | Census | TIGER boundary download |
| `NCESDownloader.download_locale_boundaries()` | NCES | 12-type locale territory polygons |
| `NCESDownloader.download_school_locations()` | NCES | School point locations with locale |
| `NLRB_REGIONS` | NLRB | Region → state mapping dict |
| `fetch_enacted_plan()` | RDH | Enacted redistricting plan boundaries |
| `compute_compactness()` | RDH | Polsby-Popper, Reock, etc. |
| `demographic_profile()` | RDH | Aggregate demographics by district |
| `build_multi_source_tabulation()` | This notebook | Unified county-level join |

### Important Caveats

- **MoE propagation**: ACS estimates carry margins of error that must be propagated
  through aggregations. Use `sqrt(sum(MoE_i^2))` for sums; see the tabulation guide.
- **Scale mismatch**: NLRB regions are state-level; NCES locales are sub-tract.
  Cross-source joins inherit the coarser resolution.
- **Areal interpolation**: Spatial overlays (tract → district) assume uniform
  population distribution within source zones — this is rarely true.
- **Temporal alignment**: Census ACS 5-year estimates span a range (e.g., 2018–2022).
  NCES data is for a single school year. Match vintages carefully.

For detailed treatment of these issues, see `docs/MULTI_SOURCE_TABULATION_GUIDE.md`.

### See Also

- **[04_Spatial_Data_Census_Boundaries.ipynb](04_Spatial_Data_Census_Boundaries.ipynb)** — TIGER, GEOID, crosswalks, time-series, isochrones
- **[15_Census_Demographics_Integration.ipynb](15_Census_Demographics_Integration.ipynb)** — Census convenience functions quick-start
- **[19_NLRB_Data_Integration.ipynb](19_NLRB_Data_Integration.ipynb)** — NLRB region model, population service
- **docs/MULTI_SOURCE_TABULATION_GUIDE.md** — MoE propagation, scale, areal interpolation theory